# Lab 1.4 &mdash; One Agent or Three: Building the Same App Twice

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 50 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Build the same capability twice: one agent with three tools, three agents with one each
- Run one eval set through both and find out which one you would actually ship
- Find the bug the second architecture introduces &mdash; a handoff that loses information
- Fix it with typed handoffs (<code>response_format</code>) and watch the accuracy come back

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Lab 1.3.** Same tools, same case file. The question is no longer whether
> an agent works &mdash; it is what breaks when you split one into three, and how you fix it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

Splitting one agent into three buys you specialisation: shorter prompts, fewer tools each, a
clearer place to put a control. It costs you **coordination** &mdash; every handoff is another model
call, another context to rebuild, another place to lose information.

The usual mistake is to assume the cost of that is a bit more latency. It is not. The expensive
part is that **every handoff is a lossy re-encoding**: the worker answers in prose, the supervisor
has to recover a fact from it, and whatever does not survive that step is silently gone. The run
still completes. The answer is still confident. It is just wrong.

So this lab builds the same capability three ways &mdash; one agent, three agents handing off in
prose, three agents handing off a typed object &mdash; and asks the only question that matters
first: **which one would you ship?** Calls, latency and tokens are printed too, because you should
know how to read them, but they are not the finding.

## Section 1 &mdash; Instrument first, argue later

Before comparing two architectures you need to see what each one *did*: how many model calls, how
many tool calls, how long, and &mdash; since this is the one lab in Module 1 that looks at cost &mdash;
how many tokens. `usage_metadata` on each `AIMessage` carries what the gateway actually reported,
so you are reading the real thing rather than estimating from string length.

Do not read too much into the token column. It is here once, so you know how to get it when you
need it. Everything that follows is about whether the thing *works*.

In [ ]:
class Meter:
    """Tokens, wall time and model calls for one architecture over one eval set."""

    def __init__(self, label: str):
        self.label = label
        self.in_tokens = self.out_tokens = self.calls = self.tool_calls = 0
        self.seconds = 0.0

    def record(self, result: dict, seconds: float) -> None:
        """Add one agent run. `result` is what create_agent returned."""
        self.seconds += seconds
        for m in result["messages"]:
            if getattr(m, "type", None) != "ai":
                continue
            self.calls += 1
            self.tool_calls += len(m.tool_calls or [])
            usage = getattr(m, "usage_metadata", None) or {}
            self.in_tokens += BLANK    # TODO: the prompt side of the bill
            self.out_tokens += usage.get("output_tokens", 0)

    @property
    def total_tokens(self) -> int:
        return self.in_tokens + self.out_tokens

    def row(self, cases: int) -> str:
        return (f"{self.label:22} {self.total_tokens:>8} tok  {self.calls:>3} calls  "
                f"{self.tool_calls:>3} tools  {self.seconds:>6.1f}s  "
                f"{self.total_tokens / max(cases, 1):>7.0f} tok/case")

In [ ]:
# --- Self-check: Section 1   (canned messages -- no model call)
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

def _fake_run(n_in, n_out, tools=0):
    ai = AIMessage(content="x", tool_calls=[{"name": "t", "args": {}, "id": f"c{i}",
                                             "type": "tool_call"} for i in range(tools)])
    ai.usage_metadata = {"input_tokens": n_in, "output_tokens": n_out, "total_tokens": n_in + n_out}
    return {"messages": [HumanMessage("q"), ai]}

def _metered():
    m = Meter("test")
    m.record(_fake_run(100, 20, tools=1), 1.5)
    m.record(_fake_run(300, 30), 2.5)
    return m

check("input tokens are counted", lambda: _metered().in_tokens == 400,
      "the context you resend is the larger half of the bill -- count it")
check("output tokens are counted", lambda: _metered().out_tokens == 50)
check("total is both sides", lambda: _metered().total_tokens == 450)
check("model calls are counted", lambda: _metered().calls == 2)
check("tool calls are counted separately", lambda: _metered().tool_calls == 1)
check("wall time accumulates", lambda: abs(_metered().seconds - 4.0) < 1e-6)
check("a run with no usage metadata does not crash",
      lambda: Meter("x").record({"messages": [AIMessage("no usage")]}, 0.1) is None)

## Section 2 &mdash; Two architectures over the same tools

**Arm 1 &mdash; one agent, three tools.** One `create_agent`, one context, one loop.

**Arm 2 &mdash; three specialists and a supervisor.** Each specialist is its own `create_agent` with
exactly one tool and a narrow prompt. The supervisor decides who to call, and the answer has to
be assembled from what comes back.

Both arms must answer the same questions. Write the supervisor's routing rule.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from pydantic import BaseModel, Field

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1003'."""
    r = LEDGER.get(ref)
    return json.dumps({"ref": ref, **r}) if r else f"no payment found with reference {ref!r}"

@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'."""
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")

@tool
def requires_human_approval(reason_code: str) -> str:
    """Return whether a reason code obliges a human decision before any action."""
    return json.dumps({"reason_code": reason_code, "needs_human": reason_code in NEEDS_HUMAN})

ALL_TOOLS = [lookup_payment, policy_for, requires_human_approval]

SPECIALISTS = {
    "ledger": ("You read the ledger. Look up the payment and report its fields verbatim. "
               "Do not interpret policy.", [lookup_payment]),
    "policy": ("You read the policy catalogue. Given a reason code, report the policy text "
               "verbatim. Do not look up payments.", [policy_for]),
    "control": ("You decide whether a human must approve. Given a reason code, report "
                "true or false and nothing else.", [requires_human_approval]),
}

WORKERS = tuple(SPECIALISTS)          # ("ledger", "policy", "control")

def route(step: str) -> str:
    """Which specialist handles this step of the investigation?

    step is one of "read_payment", "read_policy", "check_approval".
    """
    mapping = {"read_payment": "ledger", "read_policy": "policy", "check_approval": "control"}
    if step not in mapping:
        raise ValueError(f"no worker for step {step!r}")
    return BLANK                       # TODO: the worker this step belongs to

In [ ]:
# --- Self-check: Section 2   (routing + agent construction -- no model call)
def _bad_step():
    try:
        route("send_email")
        return False
    except ValueError:
        return True

check("each step routes to its specialist",
      lambda: [route(s) for s in ("read_payment", "read_policy", "check_approval")]
              == ["ledger", "policy", "control"])
check("an unknown step is refused, not guessed", _bad_step,
      "a supervisor that invents a worker is the single commonest multi-agent bug")
check("each specialist holds exactly one tool",
      lambda: all(len(tools) == 1 for _, tools in SPECIALISTS.values()))
check("between them the specialists cover every tool",
      lambda: {t.name for _, ts in SPECIALISTS.values() for t in ts}
              == {t.name for t in ALL_TOOLS})
check("each specialist prompt says what it must NOT do",
      lambda: all("not" in p.lower() or "nothing else" in p.lower()
                  for p, _ in SPECIALISTS.values()),
      "a narrow worker needs its boundary written down or it drifts wide")

## Section 3 &mdash; One eval set, both arms

Five cases. Each has a reference, and an assertion that is true of a correct answer &mdash; a
substring we require in the final text. Crude, deliberately: this is the baseline pass rate the
whole course measures against, and it has to be something you can defend.

In [ ]:
EVAL_SET = [
    {"ref": "PMT-1003", "q": "Investigate PMT-1003 and say what must happen next.",
     "must_contain": ["treasury"],  "needs_human": True},
    {"ref": "PMT-1005", "q": "Investigate PMT-1005 and say what must happen next.",
     "must_contain": ["compliance"], "needs_human": True},
    {"ref": "PMT-1002", "q": "Investigate PMT-1002 and say what must happen next.",
     "must_contain": ["retry"],     "needs_human": False},
    {"ref": "PMT-1004", "q": "Investigate PMT-1004 and say what must happen next.",
     "must_contain": ["originator", "r04"], "needs_human": False},
    {"ref": "PMT-1001", "q": "Investigate PMT-1001 and say what must happen next.",
     "must_contain": ["settled"],   "needs_human": False},
]

def passes(case: dict, answer: str) -> bool:
    """A case passes when the answer mentions any of its required terms."""
    low = (answer or "").lower()
    return BLANK                      # TODO: any required term present, or all of them?


def run_single(meter: Meter) -> list[bool]:
    """Arm 1: one agent, all three tools, one context per case."""
    agent = create_agent(
        model=get_llm(), tools=ALL_TOOLS,
        system_prompt=("You investigate payment exceptions. Procedure, in order: "
                       "1) look up the payment; 2) look up the policy for its reason code; "
                       "3) check whether it requires human approval; 4) state the next action."))
    out = []
    for case in EVAL_SET:
        t0 = time.time()
        result = agent.invoke({"messages": [("human", case["q"])]})
        meter.record(result, time.time() - t0)
        out.append(passes(case, result["messages"][-1].content))
    return out


def run_supervised(meter: Meter) -> list[bool]:
    """Arm 2: a supervisor calls three single-tool specialists and assembles the answer."""
    agents = {name: create_agent(model=get_llm(), tools=tools, system_prompt=prompt)
              for name, (prompt, tools) in SPECIALISTS.items()}

    def call(worker: str, message: str) -> str:
        t0 = time.time()
        result = agents[worker].invoke({"messages": [("human", message)]})
        meter.record(result, time.time() - t0)
        return result["messages"][-1].content

    out = []
    for case in EVAL_SET:
        ledger = call(route("read_payment"), f"Look up {case['ref']}.")
        code_ = next((c for c in POLICY if c in ledger), "NONE")
        policy = call(route("read_policy"), f"What is the policy for {code_}?")
        control = call(route("check_approval"), f"Does {code_} need human approval?")
        # the supervisor's own call: assemble, and pay for the context again
        t0 = time.time()
        final = create_agent(model=get_llm(), tools=[], system_prompt=(
            "You are the supervisor. Using only the specialist reports, state the next action "
            "in one short line.")).invoke({"messages": [("human",
                f"LEDGER: {ledger}\nPOLICY: {policy}\nCONTROL: {control}\n\n{case['q']}")]})
        meter.record(final, time.time() - t0)
        out.append(passes(case, final["messages"][-1].content))
    return out


def run_supervised_typed(meter: Meter) -> list[bool]:
    """Arm 3: the same three specialists, but the ledger worker returns a LedgerReport."""
    ledger_agent = create_agent(model=get_llm(), tools=[lookup_payment],
                                system_prompt=SPECIALISTS["ledger"][0],
                                response_format=LedgerReport)
    others = {name: create_agent(model=get_llm(), tools=tools, system_prompt=prompt)
              for name, (prompt, tools) in SPECIALISTS.items() if name != "ledger"}

    def call(worker: str, message: str) -> str:
        t0 = time.time()
        result = others[worker].invoke({"messages": [("human", message)]})
        meter.record(result, time.time() - t0)
        return result["messages"][-1].content

    out = []
    for case in EVAL_SET:
        t0 = time.time()
        rep = ledger_agent.invoke({"messages": [("human", f"Look up {case['ref']}.")]})
        meter.record(rep, time.time() - t0)
        report = rep.get("structured_response")             # a LedgerReport, not a paragraph
        code_ = extract_reason_code(report)                 # one field access
        policy = call("policy", f"What is the policy for {code_}?")
        control = call("control", f"Does {code_} need human approval?")
        t0 = time.time()
        final = create_agent(model=get_llm(), tools=[], system_prompt=(
            "You are the supervisor. Using only the specialist reports, state the next action "
            "in one short line.")).invoke({"messages": [("human",
                f"LEDGER: {report}\nPOLICY: {policy}\nCONTROL: {control}\n\n{case['q']}")]})
        meter.record(final, time.time() - t0)
        out.append(passes(case, final["messages"][-1].content))
    return out

In [ ]:
# --- Self-check: Section 3   (the scorer, on canned answers -- no model call)
_c = EVAL_SET[3]      # PMT-1004: passes on "originator" OR "r04"

check("an answer containing one required term passes",
      lambda: passes(_c, "Return to originator.") is True)
check("either term is enough",
      lambda: passes(_c, "Send it back with code R04.") is True,
      'must_contain is a list of alternatives -- "any", not "all"')
check("an answer containing none of them fails",
      lambda: passes(_c, "Escalate to Treasury.") is False)
check("the scorer is case-insensitive",
      lambda: passes(_c, "RETURN TO ORIGINATOR") is True)
check("an empty answer fails rather than crashing",
      lambda: passes(_c, "") is False)
check("every case names its expectation and its ref",
      lambda: all(c["must_contain"] and c["ref"] in LEDGER for c in EVAL_SET))

## Section 4 &mdash; Find where the quality went

You now have two supervisor arms that score badly. Before theorising, instrument.

The obvious suspect is the handoff. `run_supervised` recovers the reason code from the ledger
worker's **prose** with a substring search:

```python
code_ = next((c for c in POLICY if c in ledger), "NONE")
```

That looks fragile, and it is &mdash; but "looks fragile" is not evidence. Build the typed
alternative, then **measure whether the handoff is actually losing anything**. A contract is what
makes that measurable: you cannot assert on a paragraph, but you can assert on a field.

In [ ]:
class LedgerReport(BaseModel):
    """What the ledger specialist returns. A contract, not a paragraph."""
    ref: str = Field(description="The payment reference that was looked up")
    status: str = Field(description="One of: settled, failed, held")
    reason_code: str = Field(description="BLANK")  # TODO: what must the supervisor be able to read?
    amount: float = Field(description="The payment amount")


def extract_reason_code(report) -> str:
    """The supervisor's read step, for both handoff styles.

    report is a LedgerReport when the worker has a contract, or prose when it does not.
    """
    if isinstance(report, LedgerReport):
        return BLANK                  # TODO: read the field -- no searching, no guessing
    return next((c for c in POLICY if c in str(report)), "NONE")   # the substring version


TRUTH = {ref: (rec["reason_code"] or "NONE") for ref, rec in LEDGER.items()}

In [ ]:
# --- Self-check: Section 4   (both handoff styles, on canned worker output -- no model call)
_typed = LedgerReport(ref="PMT-1003", status="held", reason_code="LIMIT_BREACH", amount=990000.0)
_prose_verbatim    = "PMT-1003 is held with reason code LIMIT_BREACH for USD 990,000."
_prose_paraphrased = "Payment PMT-1003 is on hold because it breaches the value limit."

def _desc():
    d = LedgerReport.model_fields["reason_code"].description
    if not d or d == "BLANK":
        raise NameError("reason_code still has no description")
    return d

check("the contract carries everything the supervisor needs",
      lambda: set(LedgerReport.model_fields) == {"ref", "status", "reason_code", "amount"})
check("reason_code tells the worker exactly what to put there",
      lambda: len(_desc()) > 40 and "NONE" in _desc(),
      "a worker that invents its own spelling breaks the supervisor just as badly as prose")
check("a typed handoff reads the field",
      lambda: extract_reason_code(_typed) == "LIMIT_BREACH")
check("the substring version works when the worker quotes the code",
      lambda: extract_reason_code(_prose_verbatim) == "LIMIT_BREACH")
check("...and silently returns NONE when it paraphrases instead",
      lambda: extract_reason_code(_prose_paraphrased) == "NONE",
      "same fact, different sentence -- and nothing anywhere reports an error")
check("we know the right answer for every case, so the handoff can be scored",
      lambda: TRUTH["PMT-1001"] == "NONE" and TRUTH["PMT-1005"] == "SANCTIONS_REVIEW")

### Now measure it, instead of assuming

`_prose_paraphrased` proves the substring handoff **can** lose a fact. Whether it **does** on this
workload is a different question, and the only way to answer it is to run it.

`handoff_integrity()` puts the ledger specialist through both styles and scores the recovered
reason code against the ledger itself. Run it before you read the next section.

In [ ]:
def handoff_integrity() -> dict:
    """Does the reason code survive the handoff? Score both styles against the ledger."""
    typed_agent = create_agent(model=get_llm(), tools=[lookup_payment],
                               system_prompt=SPECIALISTS["ledger"][0],
                               response_format=LedgerReport)
    prose_agent = create_agent(model=get_llm(), tools=[lookup_payment],
                               system_prompt=SPECIALISTS["ledger"][0])
    rows, typed_ok, prose_ok = [], 0, 0
    for ref, want in TRUTH.items():
        t = typed_agent.invoke({"messages": [("human", f"Look up {ref}.")]})
        report = t.get("structured_response")
        got_t = extract_reason_code(report) if report is not None else "(no structured_response)"
        p = prose_agent.invoke({"messages": [("human", f"Look up {ref}.")]})
        got_p = extract_reason_code(p["messages"][-1].content)
        typed_ok += got_t == want
        prose_ok += got_p == want
        rows.append((ref, want, got_t, got_p))
    return {"rows": rows, "typed": typed_ok, "prose": prose_ok, "of": len(TRUTH)}

## Run it for real

Three arms plus the handoff audit: about fifty agent runs, so give this cell a minute or two.
Read it top to bottom &mdash; does it work, what it took, and then whether the handoff is to blame.

In [ ]:
if llm_ready():
    def _bakeoff():
        single_m = Meter("single agent")
        super_m  = Meter("supervisor (prose)")
        typed_m  = Meter("supervisor (typed)")
        single_r = run_single(single_m)
        super_r  = run_supervised(super_m)
        typed_r  = run_supervised_typed(typed_m)
        n = len(EVAL_SET)

        print("does it work?")
        print("  case       single   supervisor(prose)   supervisor(typed)")
        for case, a, b, c in zip(EVAL_SET, single_r, super_r, typed_r):
            f = lambda x: "pass" if x else "FAIL"
            print(f"  {case['ref']}   {f(a):8} {f(b):19} {f(c)}")
        print(f"\n  pass rate  {sum(single_r)}/{n} single   "
              f"{sum(super_r)}/{n} prose handoff   {sum(typed_r)}/{n} typed handoff")

        print("\nwhat it took")
        print("  architecture            calls   tools     time     tokens")
        for m in (single_m, super_m, typed_m):
            print(f"  {m.label:22} {m.calls:>5}   {m.tool_calls:>5}   {m.seconds:>6.1f}s   "
                  f"{m.total_tokens:>7}")

        print("\nis the handoff to blame?")
        hi = handoff_integrity()
        print("  ref        truth                typed                prose")
        for ref, want, got_t, got_p in hi["rows"]:
            print(f"  {ref}   {want:20} {got_t:20} {got_p}")
        print(f"\n  reason code recovered:  typed {hi['typed']}/{hi['of']}   "
              f"prose {hi['prose']}/{hi['of']}")
        return {"single": (single_m, single_r), "prose": (super_m, super_r),
                "typed": (typed_m, typed_r), "handoff": hi}
    MEASURED = guard(_bakeoff)

### Read the results

1. **The single agent won, and not narrowly.** On this eval set it answers four or five of the
   five; the supervisor arms typically manage nought to three, and the spread between runs is
   itself worth noticing &mdash; the split is not just worse, it is less predictable. That result is
   the point of Module 1's rubric, not a failure of the lab.

2. **The handoff is innocent.** This is the part worth slowing down for. The audit at the bottom
   scores the reason code recovered from the ledger worker, and on this workload *both* styles
   recover it &mdash; typically 5/5 and 5/5. The specialist was told to "report its fields verbatim",
   so it quotes the code, and even the substring search finds it. The fragile-looking line is not
   what is costing you the accuracy.

   Note what just happened: the obvious explanation was wrong, and one measurement was enough to
   retire it. Nothing else in this lab is as valuable as that habit.

3. **So where does it go?** Compare a failing case across the arms. The single agent holds the
   ledger record, the policy text and the approval flag **in one context** when it composes its
   answer. The supervisor holds three separate summaries, each written by a worker that could not
   see the question. Every fact needed to answer correctly is present somewhere in the system;
   no single agent ever holds them all at once. What is lost is not data &mdash; it is **context**.

   That is why the typed arm does not rescue the score either. Typing one edge makes that edge
   assertable, which is worth having and is exactly how you ruled it out above. It does nothing
   about the fragmentation, because the fragmentation is the architecture.

4. **The costs, briefly.** More agents means more calls and more wall time, always. Tokens often
   come out roughly level, because specialists get shorter prompts. If you expected the split to
   be dramatically more expensive and it was not, that is worth knowing too &mdash; the argument
   against splitting is rarely the bill.

5. **What would justify the split anyway?** Not elegance. Different credentials per tool,
   independent auditability, or one worker that has to be deployable on its own. None of those
   are visible in any of the numbers above, which is exactly why Lab 1.5's rubric asks about them
   *before* it looks at a result.

Module 5 is where the fragmentation gets a real fix: one typed state object every agent reads and
writes, instead of a relay of summaries. Keep this lab's numbers &mdash; you will score that graph
against them.

In [ ]:
score()

## Your turn

1. **Make the handoff guilty.** The audit came back clean because the ledger worker was told to
   report its fields *verbatim*. Change its prompt to "summarise the payment in one friendly
   sentence" and re-run `handoff_integrity()`. Watch the prose column collapse while the typed
   column holds. You have just reproduced, deliberately, the bug that was not there &mdash; which
   tells you what the contract is really insuring against.
2. **Give the workers the question.** Each specialist is asked its narrow sub-question and never
   sees what the user actually wanted. Pass the original question along with each worker call and
   re-run. How much of the gap closes? This is the cheap half of what Module 5 does properly.
3. **Make the failure loud.** `extract_reason_code` returns `"NONE"` when it finds nothing, and
   everything downstream carries on regardless. Change the supervisor to refuse rather than
   continue, and decide where that check belongs &mdash; in the worker, the supervisor, or the tool.
   Module 8 argues for one of the three.